# 8. Equivariant Drifting on SE(3)

Everything so far assumed that $\|\mathbf{x}-\mathbf{y}\|$ means something. For molecules it does not: a structure
and the *same* structure rotated by 90° are chemically identical and Euclidean-distant. Feed that pair to a kernel
and it reports "very dissimilar", so the drift will happily push a correct structure toward a rotated copy of
itself — a pure waste of capacity, and a bias, since the model is being asked to learn a distribution that is
invariant under a group while the field is not.

Two fixes, and this notebook builds both:

* **Align, then drift** (Drift-React, `THEORY.md` §11). Kabsch-align every generated structure onto the reference,
  compute the field in the aligned frame, rotate the result back. The field is then exactly equivariant:
  $\mathbf{V}(R\mathbf{X}+t) = R\,\mathbf{V}(\mathbf{X})$.
* **Drift in an invariant feature space.** Pairwise interatomic distances are invariant by construction, have a
  sparse analytic Jacobian, and let the generator keep producing Cartesian coordinates.

Along the way, the case that actually matters for structure prediction: **one input, one reference structure**. We
will find that naive single-positive drifting silently degenerates into plain MSE regression, that Drift-React's
augmented target set repairs the mechanism, and — this is the uncomfortable part — that its fixed point is still the
reference. Ensemble diversity under a single positive is a property of the *prior and the training budget*, not of
the equilibrium. That is the crux of `PROTENIX_DRIFTING.md` and it is better to meet it here, on ten atoms.

In [ ]:
import math, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0)
plt.rcParams["figure.dpi"] = 110
device = "cuda" if torch.cuda.is_available() else "cpu"
DT = torch.float64            # geometry checks want the headroom; training drops to float32
print("device:", device)

def rand_rotation(n=1, device=device, dtype=DT):
    """Uniform random rotation matrices via QR, determinant forced to +1."""
    A = torch.randn(n, 3, 3, device=device, dtype=dtype)
    Q, R = torch.linalg.qr(A)
    Q = Q * torch.sign(torch.diagonal(R, dim1=-2, dim2=-1)).unsqueeze(-2)
    flip = torch.linalg.det(Q) < 0
    Q[flip, :, 0] = -Q[flip, :, 0]
    return Q.squeeze(0) if n == 1 else Q

## 1. The problem, measured

A 10-atom structure, a random rotation of it, and a genuinely different structure. If the kernel cannot tell the
first pair apart from the second, coordinates are not a usable metric space.

In [ ]:
N_ATOM = 10
X0 = torch.randn(N_ATOM, 3, device=device, dtype=DT) * 1.5
X0 = X0 - X0.mean(0)

R = rand_rotation()
X_rot = X0 @ R + torch.tensor([2.0, -1.0, 0.5], device=device, dtype=DT)   # same molecule, moved
X_diff = X0 + 0.35 * torch.randn_like(X0)                                  # genuinely perturbed

def flat_dist(A, B): return (A - B).flatten().norm().item()

print(f"||X0 - X0 rotated+translated||  = {flat_dist(X0, X_rot):7.3f}   <- should be 0")
print(f"||X0 - X0 perturbed by 0.35 A|| = {flat_dist(X0, X_diff):7.3f}   <- should be > 0")
print(f"\nkernel exp(-d/tau) at tau=1.0:")
print(f"   identical-but-rotated : {math.exp(-flat_dist(X0, X_rot)):.3e}")
print(f"   genuinely different   : {math.exp(-flat_dist(X0, X_diff)):.3e}")

The kernel is confidently wrong: the *identical* structure scores as far less similar than the perturbed one. Every
downstream quantity — the affinity matrix, the ESS diagnostic, the drift direction — inherits this. And note that no
amount of $\tau$ tuning fixes it, because the ordering itself is wrong.

## 2. Kabsch alignment

Given two centred structures $\mathbf{P},\mathbf{Q}\in\mathbb{R}^{N\times3}$, the rotation minimising
$\|\mathbf{P}R-\mathbf{Q}\|_F$ is $R = U\,\mathrm{diag}(1,1,d)\,V^{\top}$ from the SVD of
$H=\mathbf{P}^{\top}\mathbf{Q} = U\Sigma V^{\top}$, with $d=\mathrm{sign}(\det(V^{\top}U^{\top}))$.

The $d$ matters. Without it the SVD will happily return an improper rotation — a **reflection** — which for a chiral
molecule aligns an L-enantiomer onto a D-enantiomer and reports RMSD 0. That is not a numerical detail; it is a
chemistry bug.

In [ ]:
def kabsch(P, Q):
    """Rotation R with P @ R ~= Q, batched over leading dims. Inputs must already be centred."""
    H = P.transpose(-1, -2) @ Q
    U, S, Vh = torch.linalg.svd(H)
    d = torch.sign(torch.linalg.det(Vh.transpose(-1, -2) @ U.transpose(-1, -2)))
    D = torch.eye(3, device=P.device, dtype=P.dtype).expand(*P.shape[:-2], 3, 3).clone()
    D[..., 2, 2] = d
    return U @ D @ Vh

def centre(X): return X - X.mean(-2, keepdim=True)

# (a) recovers a known rotation exactly
R_true = rand_rotation()
R_est = kabsch(centre(X0), centre(X0 @ R_true))
print(f"max |R_est - R_true|          = {(R_est - R_true).abs().max():.2e}")
print(f"RMSD after alignment          = {(centre(X0) @ R_est - centre(X0 @ R_true)).pow(2).sum(-1).mean().sqrt():.2e}")

# (b) refuses to use a reflection
X_mirror = X0 * torch.tensor([1.0, 1.0, -1.0], device=device, dtype=DT)
R_m = kabsch(centre(X0), centre(X_mirror))
print(f"\ndet(R) when asked to align a mirror image = {torch.linalg.det(R_m):.6f}  (must be +1)")
print(f"residual RMSD to the mirror image         = "
      f"{(centre(X0) @ R_m - centre(X_mirror)).pow(2).sum(-1).mean().sqrt():.4f}  (nonzero: correct)")
R_reflect = torch.diag(torch.tensor([1.0, 1.0, -1.0], device=device, dtype=DT))
print(f"if the reflection were allowed it would be = "
      f"{(centre(X0) @ R_reflect - centre(X_mirror)).pow(2).sum(-1).mean().sqrt():.2e}  (zero: wrong)")

## 3. The equivariant drifting field

Drift-React's construction (`THEORY.md` eq. 48), in five steps:

1. centre and Kabsch-align each generated structure $\mathbf{X}_i$ onto the reference, giving $R_i$ and
   $\tilde{\mathbf{X}}_i$;
2. build the **augmented target set** $\mathcal{T} = [\mathbf{Y}^{\star}, \tilde{\mathbf{X}}_1,\dots,\tilde{\mathbf{X}}_K]$
   and logits $L_{ij} = -\|\tilde{\mathbf{X}}_i - \mathcal{T}_j\|/\tau$;
3. joint normalisation $A_{ij} = \sqrt{\sigma_{\text{row}}(L)_{ij}\,\sigma_{\text{col}}(L)_{ij}}$;
4. split into attraction $A^{+}_i = A_{i,0}$ (the reference column) and repulsion $A^{-}_{ij}=A_{ij}$, $j\ge1$, and
   form
   $\tilde{\mathcal{V}}(\tilde{\mathbf{X}}_i) = A^{+}_i\sum_{j}A^{-}_{ij}(\mathbf{Y}^{\star}-\tilde{\mathbf{X}}_j)$;
5. rotate back, $\mathcal{V}(\mathbf{X}_i) = \tilde{\mathcal{V}}(\tilde{\mathbf{X}}_i)\,R_i^{\top}$.

Equivariance holds because the affinities $A_{ij}$ are **scalar invariants** — functions of aligned pairwise
distances — while the displacement vectors transform covariantly. Step 5 is what puts the answer back in the frame
the generator actually emitted.

In [ ]:
def se3_drift_field(X, Y_star, tau=0.5, joint=True):
    """Drift-React field. X: [K, N, 3] generated structures, Y_star: [N, 3] the single reference.
    Returns V: [K, N, 3] in the *input* frame."""
    Xc, Yc = centre(X), centre(Y_star)
    R = kabsch(Xc, Yc.expand_as(Xc))                     # [K, 3, 3]
    Xa = Xc @ R                                          # aligned copies
    T = torch.cat([Yc[None], Xa], 0)                     # [1 + K, N, 3]
    L = -(Xa[:, None] - T[None]).flatten(2).norm(dim=-1) / tau     # [K, 1 + K]
    A = torch.sqrt(L.softmax(-1) * L.softmax(-2)) if joint else L.softmax(-1)
    A_pos, A_neg = A[:, :1], A[:, 1:]                    # [K,1], [K,K]
    disp = Yc[None] - Xa                                 # [K, N, 3]
    V_aligned = A_pos[..., None] * torch.einsum("kj,jnd->knd", A_neg, disp)
    return torch.einsum("knd,kde->kne", V_aligned, R.transpose(-1, -2))

# ---- the equivariance test --------------------------------------------------------
K = 16
Y_star = torch.randn(N_ATOM, 3, device=device, dtype=DT) * 1.5
X = Y_star[None] + 0.6 * torch.randn(K, N_ATOM, 3, device=device, dtype=DT)

V = se3_drift_field(X, Y_star)
Rg, tg = rand_rotation(), torch.randn(3, device=device, dtype=DT) * 3.0
V_rot_in   = se3_drift_field(X @ Rg + tg, Y_star)                    # rotate the generator's output only
V_rot_both = se3_drift_field(X @ Rg + tg, Y_star @ Rg + tg)          # rotate the whole world

print(f"mean ||V||                                   = {V.norm(dim=-1).mean():.3e}")
print(f"residual  |V(XR+t) - V(X)R|   (input only)   = {(V_rot_in - V @ Rg).abs().max():.3e}")
print(f"residual  |V(XR+t) - V(X)R|   (whole world)  = {(V_rot_both - V @ Rg).abs().max():.3e}")
print(f"relative                                     = "
      f"{((V_rot_both - V @ Rg).norm() / V.norm()):.3e}   (float64)")

# and in float32, which is what you will actually train in
Xf, Yf = X.float(), Y_star.float()
Vf = se3_drift_field(Xf, Yf)
Vf_rot = se3_drift_field(Xf @ Rg.float() + tg.float(), Yf @ Rg.float() + tg.float())
print(f"relative residual in float32                 = "
      f"{((Vf_rot - Vf @ Rg.float()).norm() / Vf.norm()):.3e}")

A relative residual of $6\times10^{-15}$ in float64 and $2\times10^{-6}$ in float32 — floating-point noise in each.
The construction is not approximately equivariant, it is equivariant. This is worth checking in any implementation you write, because the
failure is silent: a field that is 99% equivariant still trains, just worse, and nothing in the loss curve tells you.

**Where the cost is.** One SVD per generated structure per step. For $K$ samples of $N$ atoms that is $K$ SVDs of
$3\times3$ matrices — negligible. The expensive part is $\|\tilde{\mathbf{X}}_i-\mathcal{T}_j\|$, which is
$O(K^2 N)$; at $K=64$, $N=2000$ atoms that is 8M distance terms per step, fine. The construction does *not* scale
badly with atom count.

**What it assumes.** A correspondence between atoms — Kabsch needs $\mathbf{X}_i$ and $\mathbf{Y}^{\star}$ indexed
consistently. For symmetric molecules or homomeric complexes that correspondence is only defined up to a permutation,
and you must resolve it first (this is exactly what Protenix's `SymmetricPermutation` does before its diffusion
loss). Align first, permute first, then drift.

## 4. The single-positive regime

Scientific datasets are usually $(\text{input}, \text{one reference output})$ pairs. That single positive breaks the
mean-shift field in a way that is easy to miss.

The attraction term is $\mathbf{V}^{+}(\mathbf{x}) = \sum_j \bar k(\mathbf{x},\mathbf{y}_j)(\mathbf{y}_j-\mathbf{x})$
with $\bar k = k/\sum_{j'}k$. With **one** positive, $\bar k \equiv 1$ identically — the kernel weight cancels
against its own partition function. So

$$\mathbf{V}^{+}(\mathbf{x}) = \mathbf{y}^{\star} - \mathbf{x},$$

**independent of $\tau$, independent of the kernel, independent of everything**. Drifting has silently become plain
$L_2$ regression onto the reference. Let us confirm that it really is $\tau$-inert.

In [ ]:
def naive_attraction(X, Y_star, tau):
    """Standard mean-shift attraction with a single positive: softmax over one element."""
    Xc, Yc = centre(X), centre(Y_star)
    R = kabsch(Xc, Yc.expand_as(Xc)); Xa = Xc @ R
    logits = -(Xa - Yc[None]).flatten(1).norm(dim=-1, keepdim=True) / tau     # [K, 1]
    w = logits.softmax(-1)                                                    # == 1, always
    return w[..., None] * (Yc[None] - Xa), w

print("naive single-positive attraction:")
for tau in [0.01, 0.1, 1.0, 10.0, 100.0]:
    v, w = naive_attraction(X, Y_star, tau)
    print(f"  tau={tau:7.2f}   softmax weight = {w.mean():.6f}   mean ||V+|| = {v.norm(dim=-1).mean():.6f}")

print("\nDrift-React augmented-set field (same structures):")
for tau in [0.01, 0.1, 1.0, 10.0, 100.0]:
    v = se3_drift_field(X, Y_star, tau=tau)
    print(f"  tau={tau:7.2f}   mean ||V|| = {v.norm(dim=-1).mean():.6e}")

The naive field is *bit-identical* across four orders of magnitude of $\tau$. Every kernel-related idea from
Notebooks 2 and 3 — multi-temperature summation, ESS diagnostics, the resolution/reach trade-off — is inert. You are
training an MSE regressor with extra steps.

The augmented-set field is not: putting $\mathbf{Y}^{\star}$ and the swarm in **one** softmax gives the partition
function something to normalise against, so $\tau$ recovers its meaning as "how close counts as close" *relative to
the spread of the model's own samples*. At small $\tau$ the field is dominated by whichever swarm members are nearest
the reference; at large $\tau$ every member contributes equally and the field reduces to a swarm-mean displacement.

### 4.1 But what is the fixed point?

Anti-symmetry gave ordinary drifting an equilibrium at $q=p$. Here there is no $p$ to speak of — there is one sample
of it. Setting $\tilde{\mathcal{V}}=0$ requires $\sum_j A^{-}_{ij}(\mathbf{Y}^{\star}-\tilde{\mathbf{X}}_j)=0$: the
affinity-weighted swarm mean must equal the reference. **The spread is not pinned by this condition at first order.**
So does the swarm keep its spread, or collapse?

Run the particle dynamics directly — no network, just integrate $\dot{\mathbf{X}}=\mathcal{V}(\mathbf{X})$ — and
watch three numbers: per-sample RMSD to the reference, the swarm's own spread, and the RMSD of the swarm *mean*.

In [ ]:
def integrate(joint, steps=6000, K=32, tau=0.5, step_size=0.05, seed=1, record=(0, 250, 1000, 3000, 6000)):
    torch.manual_seed(seed)
    Xp = Y_star[None] + 1.0 * torch.randn(K, N_ATOM, 3, device=device, dtype=DT)
    Yc = centre(Y_star)
    out = []
    for s in range(steps + 1):
        if s in record:
            Xa = centre(Xp) @ kabsch(centre(Xp), Yc.expand(K, -1, -1))
            out.append((s,
                        (Xa - Yc).pow(2).sum(-1).mean(-1).sqrt().mean().item(),   # per-sample RMSD
                        Xa.std(0).mean().item(),                                   # swarm spread
                        (Xa.mean(0) - Yc).pow(2).sum(-1).mean().sqrt().item()))    # RMSD of the mean
        V = se3_drift_field(Xp, Y_star, tau=tau, joint=joint)
        Xp = Xp + step_size * V / V.flatten(1).norm(dim=-1).mean().clamp_min(1e-12)
    return out

for joint, name in [(True, "joint (row x col) normalisation"), (False, "row normalisation only")]:
    print(name)
    print(f"  {'step':>6s} {'RMSD/sample':>12s} {'swarm spread':>13s} {'RMSD of mean':>13s}")
    for s, r, sp, rm in integrate(joint):
        print(f"  {s:6d} {r:12.4f} {sp:13.4f} {rm:13.4f}")
    print()

Both collapse. The joint normalisation buys roughly a factor of three in the *rate* — at step 250 the jointly
normalised swarm still holds ~70% of its initial spread while the row-normalised one has already given up ~68% of
its own, and it needs ~3000 rather than ~1000 steps to converge — but the fixed point of the single-positive field is
the reference structure, for both.

This is the most important negative result in this notebook, so state it plainly:

> **Under a single positive, ensemble diversity is not an equilibrium property.** It comes from the prior, from the
> generator's limited capacity to memorise, and from stopping before collapse. It does not come from the field.

The consequences are practical:

* You cannot tune $\tau$ or $\eta$ to "get more diversity" — you are choosing a point on a collapse trajectory.
* Any evaluation of ensemble spread is really an evaluation of *how long you trained*, unless something else pins it.
* If aleatoric diversity matters (alternative binding modes, conformational ensembles), it must be supplied
  structurally: an entropy-scaled prior (Notebook 7 §6), multiple references per input where the data allows, or an
  explicit diversity term.

Drift-React lives with this because a reaction pathway genuinely has one right answer given endpoints — the
distribution it needs is nearly a point mass. Structure prediction is the case where it bites, and
`PROTENIX_DRIFTING.md` §4 (R1) treats it as the central design risk rather than a detail.

There is a partial fix worth knowing, and it costs nothing: when a batch contains **many different inputs**, run the
repulsion across the *whole batch* rather than only within one input's swarm. Samples for input $a$ then repel
samples for input $b$, which does not pin the conditional spread but does prevent the model from mapping different
inputs to the same structure — mode collapse across conditions rather than within them.

## 5. The other route: drift in an invariant feature space

Instead of making the field equivariant, make it **invariant** by computing it on quantities that do not move under
$SE(3)$. The natural choice is the vector of pairwise interatomic distances,

$$\phi_k(\mathbf{x}) = \|\mathbf{x}_{i_k} - \mathbf{x}_{j_k}\|, \qquad k=1,\dots,\tbinom{N}{2}.$$

This is translation- and rotation-invariant by construction (and, unlike Kabsch, needs no reference to align to). Its
Jacobian is sparse and analytic:

$$\frac{\partial\phi_k}{\partial\mathbf{x}_a} =
\begin{cases}+\hat{\mathbf{r}}_{i_kj_k} & a=i_k\\ -\hat{\mathbf{r}}_{i_kj_k} & a=j_k\\ \mathbf{0}&\text{otherwise}\end{cases},
\qquad \hat{\mathbf{r}}_{ij}=\frac{\mathbf{x}_i-\mathbf{x}_j}{\|\mathbf{x}_i-\mathbf{x}_j\|}$$

— six nonzeros per row, so backpropagating the drift through $\phi$ is cheap even for large $N$. Let us verify the
analytic form against autograd, then confirm invariance.

In [ ]:
IU = torch.triu_indices(N_ATOM, N_ATOM, offset=1).to(device)

def pdist_features(X):
    """[..., N, 3] -> [..., N(N-1)/2] vector of pairwise distances."""
    return (X[..., IU[0], :] - X[..., IU[1], :]).norm(dim=-1)

def pdist_jacobian(X):
    """Analytic Jacobian [n_pairs, N, 3]; exactly 6 nonzeros per row."""
    diff = X[IU[0]] - X[IU[1]]
    rhat = diff / diff.norm(dim=-1, keepdim=True)
    J = torch.zeros(IU.shape[1], X.shape[0], 3, device=X.device, dtype=X.dtype)
    J[torch.arange(IU.shape[1]), IU[0]] = rhat
    J[torch.arange(IU.shape[1]), IU[1]] = -rhat
    return J

Xg = X0.clone().requires_grad_(True)
J_auto = torch.autograd.functional.jacobian(pdist_features, Xg)
J_ana = pdist_jacobian(X0)
print(f"max |analytic - autograd Jacobian| = {(J_auto - J_ana).abs().max():.2e}")
print(f"nonzeros per row: analytic {int((J_ana != 0).sum(dim=(1,2)).float().mean())}, "
      f"dense would be {N_ATOM * 3}")

phi = pdist_features(X0)
Rg2, tg2 = rand_rotation(), torch.randn(3, device=device, dtype=DT)
print(f"\ninvariance |phi(X) - phi(XR+t)|     = {(phi - pdist_features(X0 @ Rg2 + tg2)).abs().max():.2e}")
print(f"for comparison, |X - (XR+t)|        = {(X0 - (X0 @ Rg2 + tg2)).abs().max():.3f}")

Exact, and six nonzeros per row as promised. Running the drifting loss on $\phi$ therefore gives an invariant field
for free while the generator's output stays in Cartesian coordinates — the gradient reaches the coordinates through
the Jacobian.

**Which route to pick.** They are not equivalent:

| | align-then-drift | invariant features |
|---|---|---|
| needs atom correspondence | yes (Kabsch) | yes (pair indexing) |
| needs a reference to align to | **yes** | no |
| cost | $K$ SVDs + $O(K^2N)$ | $O(K^2N^2)$ features, sparse backward |
| what the metric measures | RMSD after optimal superposition | agreement of the full distance matrix |
| sensitive to global shape | yes | yes, but weights local contacts equally |
| chirality | preserved ($\det=+1$) | **destroyed** — mirror images have identical $\phi$ |

That last row is a real trap. Pairwise distances cannot distinguish enantiomers, so a purely distance-based drift
gives zero signal about chirality and a model can drift into a mirrored geometry with no penalty. If you go this
route, either add a chirality-sensitive feature (signed dihedrals, or the sign of a triple product on a reference
quadruple of atoms) or keep a coordinate-space term alongside. Notebook 9 §6 shows a second, subtler consequence of
working in distance space.

## 6. Hard constraints by output parameterisation

Drift-React's third transferable idea has nothing to do with kernels. A reaction pathway must *start* at the reactant
and *end* at the product. You can add a loss term for that and get approximate satisfaction, or you can make it
impossible to violate:

$$\mathbf{X}(\alpha_k) = (1-\alpha_k)\,\mathbf{X}_{\text{reactant}} + \alpha_k\,\mathbf{X}_{\text{product}}
\;+\; \underbrace{\sin(\pi\alpha_k)}_{=0\ \text{at}\ \alpha=0,1}\cdot\,\Delta_\theta(\alpha_k,\boldsymbol\epsilon)$$

The envelope vanishes at both ends, so the endpoints are exact **for every value of the network weights**, at
initialisation, during training, and at any inference temperature. The drift never has to spend capacity on a
constraint that geometry can enforce for free.

In [ ]:
N_FRAME = 12
alpha = torch.linspace(0, 1, N_FRAME, device=device, dtype=DT)

class PathwayGen(nn.Module):
    """Given endpoints and noise, emit a whole pathway in one forward pass.
    Endpoints are exact by construction via the sin(pi*alpha) envelope."""
    def __init__(self, n_atom=N_ATOM, d_z=8, h=256):
        super().__init__()
        self.d_z = d_z
        self.net = nn.Sequential(
            nn.Linear(d_z + 1 + 6 * n_atom, h), nn.SiLU(),
            nn.Linear(h, h), nn.SiLU(),
            nn.Linear(h, h), nn.SiLU(),
            nn.Linear(h, 3 * n_atom))
        # small random init: the pathway starts near the straight line but *does* depend on z
        nn.init.normal_(self.net[-1].weight, std=0.01); nn.init.zeros_(self.net[-1].bias)
    def forward(self, xr, xp, z):
        K = z.shape[0]
        ends = torch.cat([xr.flatten(), xp.flatten()])[None, None].expand(K, N_FRAME, -1)
        a = alpha.to(z.dtype)[None, :, None].expand(K, N_FRAME, 1)
        inp = torch.cat([z[:, None].expand(K, N_FRAME, self.d_z), a, ends], -1)
        delta = self.net(inp).view(K, N_FRAME, N_ATOM, 3)
        lin = (1 - a[..., None]) * xr + a[..., None] * xp
        return lin + torch.sin(math.pi * a)[..., None] * delta

torch.manual_seed(0)
x_react = torch.randn(N_ATOM, 3, device=device, dtype=DT)
x_prod = x_react + torch.randn(N_ATOM, 3, device=device, dtype=DT) * 0.8
gen = PathwayGen().to(device).to(DT)

# deliberately randomise every layer: the constraint must survive arbitrary weights
with torch.no_grad():
    for p in gen.parameters(): p.add_(torch.randn_like(p) * 0.5)

P = gen(x_react, x_prod, torch.randn(4, gen.d_z, device=device, dtype=DT))
print(f"|first frame - reactant| = {(P[:, 0] - x_react).abs().max():.2e}")
print(f"|last frame  - product | = {(P[:, -1] - x_prod).abs().max():.2e}")
print(f"interior frames displaced by {(P[:, 1:-1] - ((1 - alpha[1:-1, None, None]) * x_react + alpha[1:-1, None, None] * x_prod)).norm(dim=-1).mean():.1f} A on average")
print("   (the interior is deliberate nonsense - we added N(0, 0.5) noise to every weight.")
print("    That is exactly the point: the endpoints hold anyway.)")

Exact to floating-point precision, with randomised weights. Compare the alternative — an endpoint penalty $\lambda\|X_0-X_r\|^2$ —
which is approximate, adds a hyperparameter, competes with the drift for gradient, and gets *worse* as the drift gets
stronger.

The general principle transfers well beyond pathways: **express every hard constraint in the output parameterisation
and let the field work in the quotient space that remains.** Bond lengths via internal coordinates, unit quaternions
for rotations, simplex outputs via softmax, endpoints via an envelope. Anything the parameterisation guarantees, the
drift never has to learn — and cannot break.

### A small end-to-end run

Reference pathway: the endpoints, bowed outward along a fixed direction (a stand-in for a barrier-crossing path that
does not go straight through). One reference per input — the single-positive regime of §4 — and the equivariant field
of §3.

In [ ]:
def reference_path(xr, xp, bow=0.9):
    a = alpha[:, None, None]
    lin = (1 - a) * xr + a * xp
    d = F.normalize((xp - xr).mean(0), dim=-1)
    perp = F.normalize(torch.cross(d, torch.tensor([0.0, 0.0, 1.0], device=device, dtype=DT), dim=-1), dim=-1)
    return lin + bow * torch.sin(math.pi * alpha)[:, None, None] * perp

Y_path = reference_path(x_react, x_prod)

def train_pathway(steps=1500, K=16, tau=0.5, eta=0.4, lr=1e-3, seed=0):
    torch.manual_seed(seed)
    g = PathwayGen().to(device).to(DT)
    opt = torch.optim.AdamW(g.parameters(), lr, betas=(0.9, 0.95))
    hist = []
    for s in range(steps):
        Xp = g(x_react, x_prod, torch.randn(K, g.d_z, device=device, dtype=DT))
        Xf = Xp.reshape(K, N_FRAME * N_ATOM, 3)                 # treat the pathway as one point cloud
        Yf = Y_path.reshape(1, N_FRAME * N_ATOM, 3)[0]
        with torch.no_grad():
            V = se3_drift_field(Xf, Yf, tau=tau)
            V = V / V.flatten(1).norm(dim=-1).mean().clamp_min(1e-12)
            target = Xf + eta * V
        loss = F.mse_loss(Xf, target)
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(g.parameters(), 2.0); opt.step()
        if s % 100 == 0:
            with torch.no_grad():
                rmsd = (Xp - Y_path[None]).pow(2).sum(-1).mean().sqrt().item()
                endpt = max((Xp[:, 0] - x_react).abs().max().item(),
                            (Xp[:, -1] - x_prod).abs().max().item())
                spread = Xp.std(0).mean().item()          # does the noise input matter at all?
            hist.append((s, rmsd, endpt, spread))
    return g, hist

g_path, hist = train_pathway()
print(f"{'step':>6s} {'RMSD to reference':>18s} {'endpoint violation':>20s} {'ensemble spread':>17s}")
for s, r, e, sp in hist: print(f"{s:6d} {r:18.4f} {e:20.2e} {sp:17.4f}")

# does the trained pipeline stay equivariant end to end?
z = torch.randn(8, g_path.d_z, device=device, dtype=DT)
Rg3, tg3 = rand_rotation(), torch.randn(3, device=device, dtype=DT)
with torch.no_grad():
    P1 = g_path(x_react, x_prod, z)
    P2 = g_path(x_react @ Rg3 + tg3, x_prod @ Rg3 + tg3, z)
print(f"\ngenerator equivariance |f(XR+t) - f(X)R - t| = {(P2 - (P1 @ Rg3 + tg3)).abs().max():.2e}")

Three things to read here. The RMSD falls — the field works. The endpoint violation stays at $10^{-16}$ throughout,
which is the parameterisation doing its job. And the **ensemble spread collapses toward zero within a few hundred
steps**: §4's negative result, now with a network in the loop. The generator was initialised with a genuine (if
small) dependence on its noise input, and training removes it — it learns to ignore $\boldsymbol\epsilon$ and emit
the reference pathway.

Given a single positive per input, that is the *correct* solution to the problem as posed. Which is precisely why, if
you want an ensemble, you must pose a different problem.

The generator equivariance residual, however, is **not** small: the MLP takes raw coordinates as input and is not an
equivariant architecture. The *field* is equivariant (§3), so training is unbiased with respect to global pose, but
the trained model still has to have *learned* pose-invariance from data augmentation. In a real system you would
either use an equivariant architecture, or canonicalise the input frame, or (as AlphaFold3 and Protenix do) apply
random rotations to every training example and let the network learn approximate equivariance. An equivariant field
does not make the network equivariant; it makes the *target* well-defined.

## 7. What carries over

For any attempt to drift on 3-D structures:

1. **Align before you compare.** Kabsch with the reflection guard, resolving symmetry-related permutations first.
2. **Test equivariance numerically**, not by inspection. The residual should be at floating-point noise; anything
   larger means a frame is being dropped somewhere.
3. **One positive means no kernel.** Check that your field actually depends on $\tau$. If it does not, you are
   running MSE.
4. **The single-positive fixed point is the reference.** Diversity must be supplied by the prior or the training
   budget. Measure ensemble spread as a function of training step before believing any diversity number.
5. **Invariant features are cheap and reference-free but achiral.** Add a chirality-sensitive term if you use them.
6. **Put hard constraints in the parameterisation.** Endpoints, bond lengths, simplex outputs — anything geometry can
   guarantee, the field should never see.
7. **An equivariant field is not an equivariant model.** They are separate requirements with separate fixes.

## Summary

* Coordinates are not a metric space for molecules; a rotated copy scores as less similar than a genuinely perturbed
  structure. Kabsch alignment inside the field construction fixes this **exactly** — verified to floating-point noise
  in both float64 and float32.
* Drift-React's augmented target set restores $\tau$-dependence in the single-positive regime, where the naive
  mean-shift attraction degenerates to $\mathbf{y}^{\star}-\mathbf{x}$ regardless of the kernel.
* But its fixed point is still the reference: measured collapse of a 32-member swarm from spread 0.85 to 0.003.
  **Under one positive, ensemble diversity is a budget, not an equilibrium.**
* Pairwise-distance features give invariance for free with a 6-nonzero-per-row analytic Jacobian — at the cost of
  chirality blindness.
* A $\sin(\pi\alpha)$ envelope enforces endpoint constraints to $10^{-15}$ for arbitrary network weights. Hard
  constraints belong in the parameterisation.

Next: [9. Score and Force Guidance: Drifting toward Boltzmann](<9. Score and Force Guidance.ipynb>) — what to do when
your data is not distributed the way you want it to be.